# Embedding for Sentiment Analysis 

Now that we know how word embedding works, we'll apply it to a supervised problem of sentiment analysis. The idea is to classify the comments left by users according to the number of stars they gave the Disneyland resort park in their reviews.

## Data Preprocessing

### Import Data 

1. Import the necessary libraries

In [1]:
import io
import os
import re
import shutil
import tarfile
import string
import torch
import tiktoken
import requests
import numpy as np
import pandas as pd
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset, random_split

device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"
print(f"Using {device} device")

Using mps device


2. Copy the link below and read the file it contains with `pandas`.

* https://go.aws/314bBDq

In [2]:
# Import dataset with Pandas 


,user_id,review,stars,date_format,time_of_day,hour_of_day,day_of_week,review_format,review_lang,month_year,review_len,review_nb_words
0,efb62a167fee5cf3678b24427de8e31f,"Génial, fabuleux, exceptionnel ! J'aimerais qu...",5,2017-09-29 18:17:00,18:17,18,Ven,génial fabuleux exceptionnel j aimerais qu...,french,2017-09,115,19
1,e3be4f9c9e0b9572bfb2a5f88497bb14,NaN,2,2017-09-29 17:29:00,17:29,17,Ven,NaN,NaN,2017-09,0,0
2,1b8e5760162d867e9b9ca80f645bdc60,"Toujours aussi magic, féerique !",5,2017-09-29 16:46:00,16:46,16,Ven,toujours aussi magic féerique,french,2017-09,32,4
3,fa330e5891a1bb486c3e9bf95c098726,NaN,5,2017-09-29 15:52:00,15:52,15,Ven,NaN,NaN,2017-09,0,0
4,c1a693206aee1a2412d4bd9e45b80ec5,NaN,3,2017-09-29 15:29:00,15:29,15,Ven,NaN,NaN,2017-09,0,0


3. We will need the reviews in French. Filter the reviews so that they are in the right language. For this you need to find a column that gives you that information.

In [3]:
# Taking only french reviews


,user_id,review,stars,date_format,time_of_day,hour_of_day,day_of_week,review_format,review_lang,month_year,review_len,review_nb_words
0,efb62a167fee5cf3678b24427de8e31f,"Génial, fabuleux, exceptionnel ! J'aimerais qu...",5,2017-09-29 18:17:00,18:17,18,Ven,génial fabuleux exceptionnel j aimerais qu...,french,2017-09,115,19
2,1b8e5760162d867e9b9ca80f645bdc60,"Toujours aussi magic, féerique !",5,2017-09-29 16:46:00,16:46,16,Ven,toujours aussi magic féerique,french,2017-09,32,4
11,726b1a3e2664e8b075129bcd643dbf56,En vacances en région parisienne nous nous som...,2,2017-09-29 00:37:00,00:37,0,Ven,en vacances en région parisienne nous nous som...,french,2017-09,172,25
12,8a71763fbb3da7436b957681b24cc404,Tropbeaufinalpleinlesyeuxoreil,5,2017-09-29 00:16:00,00:16,0,Ven,tropbeaufinalpleinlesyeuxoreil,french,2017-09,30,1
23,ce7abd7798ee036d667c0ad84b85daa7,L'univers Disney reste merveilleux. Toutefois ...,4,2017-09-28 20:24:00,20:24,20,Jeu,l univers disney reste merveilleux toutefois ...,french,2017-09,148,23


4. Keep only the `review` & `stars` columns.

In [4]:
# Let's take the columns we're interested in 


,review,stars
0,"Génial, fabuleux, exceptionnel ! J'aimerais qu...",5
2,"Toujours aussi magic, féerique !",5
11,En vacances en région parisienne nous nous som...,2
12,Tropbeaufinalpleinlesyeuxoreil,5
23,L'univers Disney reste merveilleux. Toutefois ...,4


### Preprocessing

We will now go through a preprocessing phase. The goal is to convert the character strings into sequences of tokens represented by integers.

1. Use the tiktoken library in order to tokenize each sentence based on the `cl100k_base` tokenizer.

In [5]:

# print the first ten tokens of the first tokenized sentence


[38, 10610, 532, 11, 9765, 1130, 2249, 11, 4788, 8301]

2. In order to build the data loader, we need all sequences to be of the same length. Calculate the max and average senquence length, and decide which length you want all sequences to adopt.

In [6]:
# How are sequence lengths distributed?


avg seq len 88.31555345763512
max seq len 4367


3. Form a torch dataset object based on the token sequences and labels, and split the data into a train and validation set.

In [8]:
# Define a custom PyTorch dataset class for IMDB reviews
class IMDBDataset(Dataset):
    """
    A custom dataset class for IMDB reviews.

    This class is used to convert text data (already tokenized) and their corresponding labels
    into a PyTorch Dataset object, which can be easily loaded into a DataLoader.
    """

    def __init__(self, texts, labels):
        """
        Initializes the dataset by storing texts and labels as PyTorch tensors.

        Args:
        - texts (list or numpy array): Tokenized text data, where each text has been converted 
                                       into a sequence of word indices (integer tokens).
        - labels (list or numpy array): The corresponding labels for each text (e.g., sentiment scores or star ratings).
        """
        # Convert text sequences to a PyTorch tensor (long type since they are indices)

        # Convert labels to a PyTorch tensor (float32 for compatibility with loss functions)

    def __len__(self):
        """
        Returns the total number of samples in the dataset.

        This method is required for PyTorch datasets as it allows DataLoader to determine
        how many batches it needs.
        """

    def __getitem__(self, idx):
        """
        Retrieves a single data point (text and label) from the dataset based on an index.

        Args:
        - idx (int): Index of the sample to retrieve.

        Returns:
        - tuple: A tuple containing:
            - self.texts[idx]: The tokenized text at index `idx`.
            - self.labels[idx]: The corresponding label for that text.
        """

# Example usage: Creating a dataset instance

# Split dataset into training (80%) and validation (20%)

# Create DataLoaders


In [9]:
print(label)
print(text)

tensor([5., 5., 4., 4., 5., 4., 5., 5., 4., 4., 4., 4., 5., 4., 5., 5., 5., 5.,
        2., 2., 4., 3., 5., 5., 1., 5., 5., 4., 5., 5., 4., 5.])
tensor([[56948, 39892, 86323,  ...,     0,     0,     0],
        [ 3198,   635,  3869,  ...,     0,     0,     0],
        [ 5001,  2126,  1821,  ...,     0,     0,     0],
        ...,
        [ 1110,  2727,   259,  ...,     0,     0,     0],
        [ 1844,  7970, 46110,  ...,     0,     0,     0],
        [ 1844,   523,   283,  ...,    22,  4748, 12885]])


## Build the embedding based prediction model

Now that the data is duely tokenized, let's create a prediction model based on the embedding layer.

1. The first question you need to ask yourself is what kind of prediction problem are we dealing with? The target variable represents the number of stars associated with each comment.

Treating this as a regression problem seems relevant for two reasons :
- The target variable is qualitative ordinal, therefore values of stars can be compared
- This would help the model associate tokens with quantitative measures on only one dimension (as opposed to 5 dimensions in the case of classification) observations associated with each number of stars will actually benefit the training for all values of stars.

2. Build a prediction model based on your choice

In [10]:
# Get the vocabulary size from the tokenizer
# This represents the total number of unique words in the dataset,
# which will be used as the input size for the embedding layer.

# Define a neural network model for text regression
class TextRegressor(nn.Module):
    """
    A simple text regression model using embeddings and pooling.

    This model takes tokenized text as input and predicts a continuous value (e.g., sentiment score or rating).
    """

    def __init__(self, vocab_size, embed_dim):
        """
        Initializes the model layers.

        Args:
        - vocab_size (int): The number of unique words in the vocabulary.
        - embed_dim (int): The size of each word's embedding vector.

        The model consists of:
        1. An Embedding layer that converts tokenized words into dense vectors.
        2. A Pooling layer that reduces the sequence length by averaging word embeddings.
        3. A Fully Connected (Linear) layer that maps the pooled embeddings to the output value.
        """

        # Embedding layer: Maps word indices to dense vector representations
        # padding_idx=0 ensures that padding tokens (index 0) do not contribute to learning

        # Adaptive Average Pooling: Computes the average of the word embeddings along the sequence length
        # This helps reduce variable-length text into a fixed-size representation

        # Fully Connected (Linear) layer: Maps the fixed-size vector to a single output value

    def forward(self, text):
        """
        Defines the forward pass of the model.

        Args:
        - text (Tensor): A batch of tokenized text (word indices).

        Returns:
        - Tensor: The predicted output (e.g., a continuous score or rating).
        """
        # Convert input word indices into dense embeddings

        # Permute to match the expected shape for pooling: (batch, channels, sequence_length)
        # Then, apply average pooling to reduce sequence length to 1

        # Pass the pooled embeddings through the linear layer to


3. Print out the sructure of the model

In [11]:


# Print model summary



TextRegressor(
  (embedding): Embedding(100277, 16, padding_idx=0)
  (pooling): AdaptiveAvgPool1d(output_size=1)
  (fc): Linear(in_features=16, out_features=1, bias=True)
)


Layer (type:depth-idx)                   Output Shape              Param #
TextRegressor                            [32, 1]                   --
├─Embedding: 1-1                         [32, 100, 16]             1,604,432
├─AdaptiveAvgPool1d: 1-2                 [32, 16, 1]               --
├─Linear: 1-3                            [32, 1]                   17
Total params: 1,604,449
Trainable params: 1,604,449
Non-trainable params: 0
Total mult-adds (M): 51.34
Input size (MB): 0.03
Forward/backward pass size (MB): 0.41
Params size (MB): 6.42
Estimated Total Size (MB): 6.85

4. Prepare and run the training loop for 50 epochs.

In [12]:
# Define the loss function
# This function measures how well the model's predictions match the actual values.
# Mean Squared Error (MSE) is commonly used for regression problems.

# Define the optimizer
# The optimizer updates the model's weights to minimize the loss function.
# Adam is an adaptive optimization algorithm that adjusts learning rates during training.

def train(model, train_loader, val_loader, criterion, optimizer, epochs=100):
    """
    Function to train a PyTorch model with training and validation datasets.
    
    Parameters:
    model: The neural network model to train.
    train_loader: DataLoader for the training dataset.
    val_loader: DataLoader for the validation dataset.
    criterion: Loss function (e.g., Mean Squared Error for regression).
    optimizer: Optimization algorithm (e.g., Adam, SGD).
    epochs: Number of training epochs (default=100).
    
    Returns:
    history: Dictionary containing loss and metric for both training and validation.
    """
    
    # Dictionary to store training & validation loss and accuracy over epochs

    for epoch in range(epochs):  # Loop over the number of epochs
        model.train()  # Set model to training mode
        total_loss, metric = 0, 0  # Initialize total loss and correct predictions
        
        # Training loop
        for inputs, labels in train_loader:
            optimizer.zero_grad()  # Reset gradients before each batch
            # Forward pass
            # Compute loss
            # Backpropagation (compute gradients)
            # Update model parameters
            
            # Accumulate batch loss
        
        # Compute average loss and accuracy for training
        
        # Validation phase (without gradient computation)
        model.eval()  # Set model to evaluation mode
        val_loss, val_correct = 0, 0
        with torch.no_grad():  # No need to compute gradients during validation
            for inputs, labels in val_loader:
                # Forward pass
                # Compute loss
                # Accumulate validation loss
        
        # Compute average loss and accuracy for validation
        
        # Store metrics in history dictionary
        
        # Print training progress
    
    return history  # Return training history

# Train the model using the training function with defined parameters


Epoch [1/50], Loss: 16.6310, metric: 4.0781, Val Loss: 13.5403, Val metric: 3.6797
Epoch [2/50], Loss: 10.3256, metric: 3.2133, Val Loss: 7.4661, Val metric: 2.7324
Epoch [3/50], Loss: 6.5428, metric: 2.5579, Val Loss: 5.9558, Val metric: 2.4405
Epoch [4/50], Loss: 5.7027, metric: 2.3880, Val Loss: 5.4386, Val metric: 2.3321
Epoch [5/50], Loss: 5.1963, metric: 2.2795, Val Loss: 4.9602, Val metric: 2.2272
Epoch [6/50], Loss: 4.7134, metric: 2.1710, Val Loss: 4.4894, Val metric: 2.1188
Epoch [7/50], Loss: 4.2410, metric: 2.0594, Val Loss: 4.0343, Val metric: 2.0086
Epoch [8/50], Loss: 3.7898, metric: 1.9467, Val Loss: 3.6091, Val metric: 1.8998
Epoch [9/50], Loss: 3.3752, metric: 1.8372, Val Loss: 3.2319, Val metric: 1.7978
Epoch [10/50], Loss: 3.0068, metric: 1.7340, Val Loss: 2.9091, Val metric: 1.7056
Epoch [11/50], Loss: 2.6927, metric: 1.6409, Val Loss: 2.6425, Val metric: 1.6256
Epoch [12/50], Loss: 2.4278, metric: 1.5582, Val Loss: 2.4226, Val metric: 1.5565
Epoch [13/50], Loss: 2

## Error analysis

Error analysis consists in focusing on the observations in the training set and validation sets that were predicted the worst by the model. This often reveals potential inconsistencies in the data, and helps identifies improvement opportunies for our model.

1. Create a function that creates a Dataframe containing:
    - the prediction value
    - the true label of the observation
    - the tokenized input
    - the text input
Based on a data loader, the tokenizer, and the model.
Apply this function to both the train loader, and the val loader.

In [31]:
# Function to evaluate the model and get worst predictions
def evaluate_worst_predictions(model, dataloader, tokenizer, device="cpu"):
    # Set model to evaluation mode to disable dropout and batch normalization

    # Lists to store all predictions, labels, errors, and inputs for analysis

    # No gradients needed during evaluation for efficiency
    with torch.no_grad():
        for batch in dataloader:
            # Extract inputs and labels from the batch
            # Move inputs and labels to the specified device (CPU/GPU)

            # Forward pass: Get model predictions

            # Convert outputs to predicted class for classification problems
            # If multi-class classification, take the class with the highest probability
            # Compute error by checking if the predicted class is incorrect

            # For regression problems, compute absolute error between predictions and true labels

            # Store predictions, labels, errors, and raw inputs for further analysis

    # Convert stored results into a Pandas DataFrame for easy analysis

    # Decode tokenized text back into human-readable text

    # Sort the DataFrame by highest error to identify the worst predictions

    # Return the sorted DataFrame containing worst predictions

# Example usage:
# Evaluate worst predictions on validation set

# Evaluate worst predictions on training set


2. Display the first ten rows of each dataframe to get an idea of the worst predicted data points. Is there anything that raises questions?

,True_Label,Predicted,Error,Inputs,Text
5834,1.0,4.450919,3.450919,"[34, 1826, 653, 842, 69596, 4809, 588, 4618, 2...",C est un endroit merveilleux!!!!!!!!!!!!!!!!!!...
2372,1.0,4.234325,3.234325,"[1844, 72006, 13612, 321, 11, 6316, 39892, 863...","Un beau soleil, une belle journée … mais une i..."
5216,1.0,4.114583,3.114583,"[47696, 708, 404, 11, 0, 0, 0, 0, 0, 0, 0, 0, ...","Bonsoir,!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!..."
1886,1.0,4.090867,3.090867,"[3198, 635, 3930, 45740, 514, 296, 37489, 220,...",Visite du parc le mardi 20 juin 2017 (journée ...
3619,1.0,4.076789,3.076789,"[53, 969, 3904, 55455, 264, 13510, 2307, 0, 0,...",Vraiment rien a dire super!!!!!!!!!!!!!!!!!!!!...
1211,1.0,4.069569,3.069569,"[56948, 40574, 978, 481, 86323, 1522, 8047, 11...","Une agréable journée passée, la rencontre avec..."
1410,1.0,4.033348,3.033348,"[73, 364, 1955, 285, 264, 834, 3520, 4363, 514...",j 'etais a disney land le 26.03.2015 avec ma f...
4319,1.0,3.946424,2.946424,"[25797, 4502, 503, 17258, 100063, 2145, 2058, ...",Ah si j pouvais visiter ce magique lieu!!!!!!!...
2852,1.0,3.862368,2.862368,"[27887, 1030, 12844, 978, 11, 21965, 409, 3890...","Écoeuré, trop de monde malgré la féerie !!!!!!..."
5203,1.0,3.835204,2.835204,"[56948, 86323, 28201, 758, 0, 0, 0, 0, 0, 0, 0...",Une journée horrible !!!!!!!!!!!!!!!!!!!!!!!!!...


,True_Label,Predicted,Error,Inputs,Text
493,1.0,5.688231,4.688231,"[87993, 79884, 758, 366, 1347, 9883, 3744, 404...",Déçu ! <br/> Partir entre amis à Disney pour p...
1158,3.0,-0.648981,3.648981,"[4643, 13281, 272, 1826, 25945, 23008, 5019, 6...","79 € c est très cher pour un parc, surtout qua..."
1560,1.0,4.406101,3.406101,"[34, 460, 267, 22761, 2428, 1880, 88661, 1143,...",Catastrophique et absolument lamentable !!!<br...
1279,1.0,4.223248,3.223248,"[1951, 2249, 49301, 220, 975, 1880, 220, 868, ...","Deux jours 14 et 15, juillet hôtel cheyenne ri..."
1384,1.0,4.152661,3.152661,"[1305, 12416, 14707, 1174, 389, 264, 81621, 40...","Très bien , on a passé de bon temps!!!!!!!!!!!..."
591,4.0,0.948650,3.051350,"[47696, 11, 3900, 308, 59858, 264, 6502, 220, ...","Bon, il n'y a pas 3,5 étoiles... alors j'arron..."
1110,1.0,4.037420,3.037420,"[43, 2908, 481, 13, 362, 961, 20028, 1208, 732...",Lamentable. A part faire la queue toute la jou...
1682,1.0,4.023272,3.023272,"[83, 897, 23008, 5019, 11083, 594, 40751, 5019...",trop cher pour mes ressources pourtant aimerai...
309,1.0,3.925804,2.925804,"[11342, 88089, 1826, 1621, 3089, 40096, 276, 4...","Mon commentaire est """""""" Momentanément INDISPO..."
62,1.0,3.905939,2.905939,"[2356, 45740, 1826, 42676, 27584, 40970, 58482...",Le parc est quand même moins féérique quand il...


3. Calculate the mean error for each category of the target. Also calculate the number of samples belonging to each category. What do you think?

Train set prediction error by star review


True_Label
1.0    1.070186
2.0    0.651015
3.0    0.487827
4.0    0.404951
5.0    0.391468
Name: Error, dtype: float32

Train set star distribution


True_Label
5.0    3908
4.0    1248
3.0     791
1.0     437
2.0     395
Name: count, dtype: int64

Validation set prediction error by star review


True_Label
1.0    1.730086
2.0    1.018811
3.0    0.761320
4.0    0.599052
5.0    0.564004
Name: Error, dtype: float32

Validation set star distribution


True_Label
5.0    973
4.0    290
3.0    219
1.0    121
2.0     92
Name: count, dtype: int64